### Import Dependencies

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types

from utils.tools import check_warehouse_availability, reserve_warehouse_items

### ADK Agent

In [ ]:
model = LiteLlm(
    model="openai/gpt-4.1-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [ ]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[check_warehouse_availability, reserve_warehouse_items],
    description="A agent that can check the availability of items in the warehouses and reserve them.",
    instruction="""
You are a part of the shopping assistant that can manage available inventory in the warehouses.

You will be given a conversation history and a list of tools, your task is to perform actions requested by the latest user query. Answer part of the query that you can answer with the available tools.

Instructions:
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combinethe required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
"""
)

### ADK Session

In [ ]:
session_service = InMemorySessionService()

In [ ]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="session_1"
)

In [ ]:
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app",
)

In [ ]:
message = types.Content(
    role="user",
    parts=[types.Part(text="What is the availability of B09WCL37Z4 in all of the warehouses?")]
)

In [ ]:
result = runner.run(
    user_id="user_1",
    session_id="session_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=3,
    )
)

In [ ]:
for event in result:
    print("===========================================")
    print(event)

In [ ]:
session_service = InMemorySessionService()

async def ask_warehouse_agent(query: str, session_id: str):

    # Check if session exists, only create if it doesn't
    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id,
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id,
        )

    runner = Runner(
        agent=warehouse_agent,
        app_name="warehouse_app",
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)],
    )

    final_text = ""

    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=3,
        ),
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text
            break

    return final_text

In [ ]:
answer = await ask_warehouse_agent("What is the availability of B09WCL37Z4 in all of the warehouses?", session_id = "123")

In [ ]:
print(answer)

In [ ]:
answer = await ask_warehouse_agent("Can you reserve 6 in Lyon", session_id = "123")

In [ ]:
answer